# Surface Normals — Baseline vs LoRA (Catastrophic Forgetting)
**4M-21 XL** — LoRA trained on depth (NYUv2, rank=8), evaluated on surface normals (DIODE val).

**Strategy**: load one model at a time to avoid OOM.
1. Baseline → 10 inferences → save to CPU → unload
2. LoRA → 10 inferences → save to CPU → unload
3. Plot grid: RGB | GT Normal | Baseline | LoRA

In [1]:
import os
REPO_ROOT = '/home/fcolasurdo/4m-peft'
os.chdir(REPO_ROOT)

FM_MODEL     = 'EPFL-VILAB/4M-21_XL'
TOK_NORMAL   = 'EPFL-VILAB/4M_tokenizers_normal_8k_224-448'
TOK_TEXT     = 'ml-4m/fourm/utils/tokenizer/trained/text_tokenizer_4m_wordpiece_30k.json'
HF_CACHE     = '/scratch-shared/tesr125338/hf_cache'
LORA_WEIGHTS = 'output/depth/nyu_rgb_r8/lora_weights-epoch49.pth'
LORA_RANK    = 8
LORA_MODULES = 'attention'
DIODE_ROOT   = '/scratch-shared/tesr125338/diode/val'
OUTPUT_DIR   = 'output/eval_forgetting'

N_SAMPLES    = 10    # images to visualise
DIVAE_STEPS  = 25
SEED         = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config OK')

Config OK


In [2]:
import sys, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as T

sys.path.insert(0, os.path.join(REPO_ROOT, 'ml-4m'))

from fourm.models.fm import FM
from fourm.vq.vqvae import DiVAE
from fourm.models.generate import (
    GenerationSampler,
    build_chained_generation_schedules,
    init_full_input_modality,
    init_empty_target_modality,
)
from fourm.data.modality_info import MODALITY_INFO
from fourm.models.lora_utils import inject_trainable_LoRA, get_LoRA_module_names
from tokenizers import Tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

/home/fcolasurdo/.conda/envs/4m-peft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


xFormers not available
xFormers not available
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [20]:
# ── Transforms ────────────────────────────────────────────────────────────────
RGB_TRANSFORM = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_rgb(path):
    """Returns (H,W,3) float [0,1] numpy array and (1,3,224,224) tensor."""
    img = Image.open(path).convert('RGB')
    tensor = RGB_TRANSFORM(img).unsqueeze(0).to(device)
    rgb_np = np.array(img.resize((224, 224))) / 255.0
    return rgb_np, tensor

# ── GT normal loading (same as eval_forgetting.py) ────────────────────────────
def load_gt_normal(normal_path, mask_path):
    """Returns gt_224 (224,224,3) unit vectors and valid (224,224) bool mask."""
    gt   = np.load(normal_path).astype(np.float32)   # (H,W,3)
    mask = np.load(mask_path).astype(bool)
    if mask.ndim == 3:
        mask = mask.squeeze(-1)

    # resize via PIL (nearest, preserves direction encoding)
    gt_pil = Image.fromarray(((gt + 1) / 2 * 255).clip(0, 255).astype(np.uint8))
    gt_224 = np.array(gt_pil.resize((224, 224), Image.NEAREST)).astype(np.float32)
    gt_224 = gt_224 / 127.5 - 1.0                    # back to [-1, 1]
    norm   = np.linalg.norm(gt_224, axis=-1, keepdims=True)
    gt_224 = gt_224 / (norm + 1e-8)                  # unit vectors

    mask_224 = np.array(
        Image.fromarray(mask.astype(np.uint8) * 255).resize((224, 224), Image.NEAREST)
    ) > 0
    valid = mask_224 & (norm.squeeze(-1) > 0.01)
    return gt_224, valid

# ── Generation params ─────────────────────────────────────────────────────────
GEN_PARAMS = dict(
    tokens_per_target       = 196,
    autoregression_scheme   = 'roar',
    decoding_steps          = 25,
    token_decoding_schedule = 'linear',
    temp                    = 1.0,
    temp_schedule           = 'constant',
    cfg_scale               = 2.0,
    cfg_schedule            = 'constant',
)

def run_normal_inference(sampler, rgb_tensor, tok_text_obj, seed=42):
    """RGB tensor → predicted normal map (224,224,3) unit vectors [CPU numpy]."""
    ntoks   = GEN_PARAMS['tokens_per_target']
    tok_mod = 'tok_normal@224'
    schedule = build_chained_generation_schedules(
        cond_domains=['rgb@224'], target_domains=[tok_mod],
        tokens_per_target=[ntoks],
        autoregression_schemes=[GEN_PARAMS['autoregression_scheme']],
        decoding_steps=[GEN_PARAMS['decoding_steps']],
        token_decoding_schedules=[GEN_PARAMS['token_decoding_schedule']],
        temps=[GEN_PARAMS['temp']], temp_schedules=[GEN_PARAMS['temp_schedule']],
        cfg_scales=[GEN_PARAMS['cfg_scale']], cfg_schedules=[GEN_PARAMS['cfg_schedule']],
        cfg_grow_conditioning=True,
    )
    eos_id = tok_text_obj.token_to_id('[EOS]')
    sd = {'rgb@224': {'tensor': rgb_tensor}}
    sd = init_full_input_modality(sd, MODALITY_INFO, 'rgb@224', device, eos_id=eos_id)
    sd = init_empty_target_modality(sd, MODALITY_INFO, tok_mod, 1, ntoks, device)
    out = sampler.generate(sd, schedule, text_tokenizer=tok_text_obj, verbose=False, seed=seed)
    return out, tok_mod

def decode_normals(out, tok_mod, tok_normal_obj):
    tokens = out[tok_mod]['ids'].squeeze(0).reshape(1, 14, 14)
    with torch.no_grad():
        dec = tok_normal_obj.decode_tokens(
            tokens, timesteps=DIVAE_STEPS, image_size=224, verbose=False
        )
    n    = dec.squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.float32)
    norm = np.linalg.norm(n, axis=-1, keepdims=True)
    return n / (norm + 1e-8)

def mae_degrees(pred_n, gt_n, valid):
    dot = np.clip((pred_n[valid] * gt_n[valid]).sum(-1), -1.0, 1.0)
    return float(np.degrees(np.arccos(dot)).mean())

def normal_to_rgb(n):
    """Normal vectors [-1,1]³ → display image [0,1]."""
    return np.clip((n + 1) / 2, 0, 1)

print('Helpers defined.')

Helpers defined.


In [5]:
# ── Collect DIODE samples ─────────────────────────────────────────────────────
samples = []
for rgb_p in sorted(Path(DIODE_ROOT).rglob('*.png')):
    normal_p = rgb_p.parent / (rgb_p.stem + '_normal.npy')
    mask_p   = rgb_p.parent / (rgb_p.stem + '_depth_mask.npy')
    if normal_p.exists() and mask_p.exists():
        split = 'indoors' if 'indoors' in str(rgb_p) else 'outdoors'
        samples.append(dict(rgb=rgb_p, normal=normal_p, mask=mask_p, split=split))

print(f'Total samples: {len(samples)}')
print(f'  indoors:  {sum(1 for s in samples if s["split"]=="indoors")}')
print(f'  outdoors: {sum(1 for s in samples if s["split"]=="outdoors")}')

# Pick N_SAMPLES: mix indoors + outdoors for visual variety
import random
random.seed(SEED)
indoor  = [s for s in samples if s['split'] == 'indoors']
outdoor = [s for s in samples if s['split'] == 'outdoors']
viz_samples = random.sample(indoor, N_SAMPLES // 2) + random.sample(outdoor, N_SAMPLES // 2)
random.shuffle(viz_samples)
print(f'\nSelected {len(viz_samples)} samples ({sum(1 for s in viz_samples if s["split"]=="indoors")} indoor, '
      f'{sum(1 for s in viz_samples if s["split"]=="outdoors")} outdoor)')

Total samples: 771
  indoors:  325
  outdoors: 446

Selected 10 samples (5 indoor, 5 outdoor)


In [6]:
# ── Load tokenizer + text tokenizer (kept alive the whole time — small) ───────
print('Loading normal tokenizer ...')
tok_normal = DiVAE.from_pretrained(TOK_NORMAL, cache_dir=HF_CACHE).to(device).eval()
tok_text   = Tokenizer.from_file(TOK_TEXT)
print('Tokenizer loaded.')

# ── Pre-load GT data (CPU only, no GPU) ───────────────────────────────────────
gt_data = []   # list of dicts: rgb_np, gt_n, valid
for s in tqdm(viz_samples, desc='Loading GT'):
    rgb_np, rgb_t = load_rgb(s['rgb'])
    gt_n, valid   = load_gt_normal(s['normal'], s['mask'])
    gt_data.append(dict(
        rgb_np  = rgb_np,
        rgb_t   = rgb_t.cpu(),   # keep on CPU — moved to GPU per-inference
        gt_n    = gt_n,
        valid   = valid,
        split   = s['split'],
        name    = s['rgb'].name,
    ))
print(f'GT loaded for {len(gt_data)} samples.')

Loading normal tokenizer ...
Tokenizer loaded.


Loading GT: 100%|██████████| 10/10 [00:02<00:00,  4.48it/s]

GT loaded for 10 samples.


In [15]:
# ── STEP 1: Baseline inference ────────────────────────────────────────────────
print('Loading baseline FM (bfloat16) ...')
fm_base = FM.from_pretrained(FM_MODEL, cache_dir=HF_CACHE).to(torch.bfloat16).to(device).eval()
sampler_base = GenerationSampler(fm_base)
print(f'  Loaded. VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

preds_base = []
for i, d in enumerate(tqdm(gt_data, desc='Baseline')):
    rgb_t = d['rgb_t'].to(device, dtype=torch.bfloat16)
    with torch.no_grad():
        out, tok_mod = run_normal_inference(sampler_base, rgb_t, tok_text, seed=SEED + i)
    pred = decode_normals(out, tok_mod, tok_normal)
    preds_base.append(pred)   # (224,224,3) numpy, CPU

# Free baseline model
del fm_base, sampler_base
torch.cuda.empty_cache(); gc.collect()
print(f'Baseline done. VRAM after unload: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading baseline FM (bfloat16) ...
  Loaded. VRAM used: 19.56 GB


Baseline: 100%|██████████| 10/10 [00:24<00:00,  2.42s/it]


Baseline done. VRAM after unload: 1.17 GB


In [17]:
# ── STEP 2: LoRA inference ────────────────────────────────────────────────────
print('Loading LoRA FM (bfloat16) ...')
fm_lora = FM.from_pretrained(FM_MODEL, cache_dir=HF_CACHE)   # float32
target_mods = get_LoRA_module_names(LORA_MODULES)
inject_trainable_LoRA(fm_lora, rank=LORA_RANK, scale=1.0,
                      target_replace_modules=target_mods)     # adds layer LoRA float32
lora_sd = torch.load(LORA_WEIGHTS, map_location='cpu')
missing, _ = fm_lora.load_state_dict(lora_sd, strict=False)
print(f'  LoRA tensors loaded: {len(lora_sd)} | missing: {len(missing)}')
fm_lora = fm_lora.to(torch.bfloat16).to(device).eval()       # converts EVERYTHING 
sampler_lora = GenerationSampler(fm_lora)
print(f'  VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

preds_lora = []
for i, d in enumerate(tqdm(gt_data, desc='LoRA')):
    rgb_t = d['rgb_t'].to(device, dtype=torch.bfloat16)
    with torch.no_grad():
        out, tok_mod = run_normal_inference(sampler_lora, rgb_t, tok_text, seed=SEED + i)
    pred = decode_normals(out, tok_mod, tok_normal)
    preds_lora.append(pred)

# Free LoRA model
del fm_lora, sampler_lora
torch.cuda.empty_cache(); gc.collect()
print(f'LoRA done. VRAM after unload: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading LoRA FM (bfloat16) ...
  LoRA tensors loaded: 336 | missing: 733
  VRAM used: 19.66 GB


LoRA: 100%|██████████| 10/10 [00:30<00:00,  3.04s/it]


LoRA done. VRAM after unload: 10.43 GB


In [18]:
# ── STEP 3: Visualisation ─────────────────────────────────────────────────────
#
# Grid: N_SAMPLES rows × 4 columns
#   col 0: RGB input
#   col 1: GT Normal (ground truth)
#   col 2: Baseline prediction  (MAE in degrees)
#   col 3: LoRA prediction      (MAE + better/worse indicator)

fig, axes = plt.subplots(N_SAMPLES, 4, figsize=(14, N_SAMPLES * 3.2))
fig.suptitle(
    'Surface Normals — DIODE val\n'
    '4M-21 XL Baseline  vs  LoRA depth rank=8 epoch49\n'
    'Colour encoding: R=X (left/right), G=Y (up/down), B=Z (depth)',
    fontsize=11, y=1.01
)

COL_LABELS = ['RGB Input', 'GT Normal', 'Baseline', 'LoRA (ep49)']
for col, lbl in enumerate(COL_LABELS):
    axes[0, col].set_title(lbl, fontsize=11, fontweight='bold', pad=6)

for row, (d, pred_b, pred_l) in enumerate(zip(gt_data, preds_base, preds_lora)):
    gt_n  = d['gt_n']
    valid = d['valid']

    mae_b = mae_degrees(pred_b, gt_n, valid)
    mae_l = mae_degrees(pred_l, gt_n, valid)
    better = mae_l < mae_b

    # col 0: RGB
    axes[row, 0].imshow(d['rgb_np'])
    axes[row, 0].axis('off')
    axes[row, 0].set_ylabel(d['split'], fontsize=8, rotation=0, labelpad=40, va='center')

    # col 1: GT
    axes[row, 1].imshow(normal_to_rgb(gt_n))
    axes[row, 1].axis('off')

    # col 2: Baseline
    axes[row, 2].imshow(normal_to_rgb(pred_b))
    axes[row, 2].axis('off')
    axes[row, 2].set_xlabel(f'MAE = {mae_b:.1f}°',
                            fontsize=9, color='#1a6bb5', fontweight='bold')

    # col 3: LoRA
    axes[row, 3].imshow(normal_to_rgb(pred_l))
    axes[row, 3].axis('off')
    arrow  = '▲ better' if better else '▼ worse'
    colour = '#2a9d2a'  if better else '#c0392b'
    delta  = mae_l - mae_b
    axes[row, 3].set_xlabel(
        f'MAE = {mae_l:.1f}°  ({arrow} {abs(delta):.1f}°)',
        fontsize=9, color=colour, fontweight='bold'
    )

plt.tight_layout()

out_path = os.path.join(OUTPUT_DIR, 'normals_viz.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

Saved → output/eval_forgetting/normals_viz.png


In [21]:
# ── Summary table ──────────────────────────────────────────────────────────────
mae_bases = [mae_degrees(pb, d['gt_n'], d['valid']) for pb, d in zip(preds_base, gt_data)]
mae_loras = [mae_degrees(pl, d['gt_n'], d['valid']) for pl, d in zip(preds_lora, gt_data)]

print('='*55)
print(f'{"Sample":<28} {"Baseline":>10} {"LoRA":>10} {"Δ":>6}')
print('-'*55)
for d, mb, ml in zip(gt_data, mae_bases, mae_loras):
    delta = ml - mb
    sign  = '+' if delta >= 0 else ''
    tag   = '▼' if delta > 0 else '▲'
    print(f'{d["name"][:25]:<28} {mb:>9.2f}° {ml:>9.2f}° {sign}{delta:>+.2f}° {tag}')
print('-'*55)
print(f'{"MEAN":<28} {np.mean(mae_bases):>9.2f}° {np.mean(mae_loras):>9.2f}° '
      f'{np.mean(mae_loras)-np.mean(mae_bases):>+.2f}°')
print('='*55)

Sample                         Baseline       LoRA      Δ
-------------------------------------------------------
00020_00185_indoors_340_0       128.43°    129.85° ++1.42° ▼
00020_00186_indoors_080_0       173.27°    172.83° -0.44° ▲
00020_00185_indoors_170_0        85.30°     90.66° ++5.36° ▼
00024_00201_outdoor_090_0       130.09°    131.61° ++1.53° ▼
00022_00194_outdoor_120_0       128.11°    131.54° ++3.43° ▼
00022_00194_outdoor_000_0       146.46°    146.59° ++0.13° ▼
00019_00183_indoors_300_0       127.36°    130.16° ++2.80° ▼
00024_00201_outdoor_080_0       120.05°    121.44° ++1.39° ▼
00019_00183_indoors_080_0       158.74°    161.62° ++2.88° ▼
00023_00200_outdoor_100_0       152.51°    150.57° -1.94° ▲
-------------------------------------------------------
MEAN                            135.03°    136.69° +1.65°
